In [0]:
dbutuls.notebook.run(./create_table_utillity)

#### Reading flight data using autoloader

In [0]:
spark.conf.set("spark.sql.legacy.timeParserPolicy","LEGACY")
from pyspark.sql.functions import to_date, current_timestamp, col, count


# Reading data
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/dbfs/FileStore/tables/schema/airport")
    .load("/mnt/geeks/rw_adls/AIRPORT/")
)

df = df.withColumn("Date_Part", to_date(current_timestamp()))


display(df)

In [0]:

from pyspark.sql.functions import split, col, to_date, expr, regexp_extract

pattern = r'^(.*?),(.*?):(.*)'

df_base = df.select(
    col("Code").alias("code"),
    regexp_extract("Description", pattern, 1).alias("city"),
    regexp_extract("Description", pattern, 2).alias("country"),
    regexp_extract("Description", pattern, 3).alias("airport"),
    to_date(col("Date_Part"), "yyyy-MM-dd").alias("Date_Part")
)

df_cleaned = df_base.filter(
    (col("city") != "") & (col("country") != "") & (col("airport") != "")
)


display(df_cleaned)



In [0]:
# Writing data
df_cleaned.writeStream.trigger(once=True).format("delta").option(
    "checkpointLocation", "/dbfs/FileStore/tables/checkpointLocation/airport"
).start("/mnt/geeks/cld_adls/airport")

#### Creating delata table on the data

In [0]:
f_delta_cleansed_load('airport', 'abfss://cleansed@geekadlsstoragesink.dfs.core.windows.net/airport', 'cleansed_geekcoders')

In [0]:
%sql
select * from cleansed_geekcoders.airport;